In [ ]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [ ]:
!pip install tqdm

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# # Load data
# train = pd.read_csv('/content/drive/MyDrive/MBZ/Thesis/bert with features/train_data_with_allfeats.csv')
# test = pd.read_csv('/content/drive/MyDrive/MBZ/Thesis/bert with features/test_data_with_allfeats.csv')
# dev = pd.read_csv('/content/drive/MyDrive/MBZ/Thesis/bert with features/dev_data_with_allfeatures_final.csv')
data_path = '/content/drive/MyDrive/MBZ/Thesis/bert with features/data_with_all_features_novocab.csv' # DATA PATH

all_df = pd.read_csv(data_path, header=0)
all_df = all_df[all_df['RL_num_19']>=12]

In [ ]:

all_df_grouped = all_df.groupby('Split')

# all_df = all_df[[DATA_COLUMN, LABEL_COLUMN]]
# all_df.columns = [DATA_COLUMN, LABEL_COLUMN]

train_df = all_df_grouped.get_group('Train')
#train_df = train_df.head(4565)
dev_df = all_df_grouped.get_group('Dev')
test_df = all_df_grouped.get_group('Test')
#tune_df = all_df.get_group('Tune')

In [ ]:
import datasets
from datasets import load_dataset


In [ ]:
train = datasets.Dataset.from_pandas(train_df)
dev = datasets.Dataset.from_pandas(dev_df)
test = datasets.Dataset.from_pandas(test_df)
#tune = datasets.Dataset.from_pandas(tune_df)
dataset = load_dataset("labr") #dump loading .. only to match the dataset template from huggingface
dataset['train'] = train
#dataset['tune'] = tune
dataset['dev'] = dev
dataset['test'] = test

In [ ]:
train_texts = dataset['train']['word_sents']
train_labels = dataset['train']['RL_num_19']

dev_texts = dataset['dev']['word_sents']
dev_labels = dataset['dev']['RL_num_19']

test_texts = dataset['test']['word_sents']
test_labels = dataset['test']['RL_num_19']
# Choose text and label
# texts = df["word_sents"].tolist()
# labels = df["RL_num_19"].astype(int).tolist()

# Drop columns not part of the feature set
non_feature_cols = ['ID', 'Sentence', 'Word Count','Readability Level', 'RL_num_19', 'Annotator', 'annotation file', 'Project Phase', 'Source', 'Book', 'Author', 'Domain', 'Text Class', 'Reader Class', 'File', 'Split',
       'Clean_Sentnece', 'max word count in RL',
       'RL_num_7', 'RL_num_5', 'RL_num_3', 'd3lex_sents', 'lex_sents',
       'd3tok_sents', 'word_sents', 'syllable_word', 'exception_lex', 'coordinated_verbs']
# relevant_columns =  ['jar_majroor', 'relative_pronoun_singular', 'negation_particle', 'verb_past_s_p', 'coordinated_verbs', 'verb_command', 'word count unique', 'broken_plural', 'verbal_sentence_with_two_objects', 'vocative', 'verbal_present_sentence_with_an_almasdariya', 'suf_pron', 'plural_masc', 'verb_present_plural', 'inna_wa_akhawataha', 'plural_fem_noun_adj', 'kana_wa_akhawataha', 'interrogative_alif', 'syllables', 'passive_voice', 'relative_pronoun_dual_plural', 'verb_past_present_dual', 'advanced_khabar', 'amma_lakin'] #[col for col in data.columns if col not in irrelevant_columns and col != 'RL_num_19']

# relevant_columns = all columns - non_feature_cols
relevant_columns = [col for col in all_df.columns if col not in non_feature_cols]
# remove coordinated_verbs, exception_lex
train_features = train_df[relevant_columns]
train_features = train_features.astype(float).values

dev_features = dev_df[relevant_columns]
dev_features = dev_features.astype(float).values

test_features = test_df[relevant_columns]
test_features = test_features.astype(float).values

# feature_df = all_df.drop(columns=non_feature_cols)
# features = feature_df.astype(float).values
feature_dim = train_features.shape[1]

train_labels = [label - 12 for label in train_labels]
dev_labels = [label - 12 for label in dev_labels]
test_labels = [label - 12 for label in test_labels]

In [ ]:
feature_dim

52

In [ ]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
encodings = tokenizer(train_texts, padding=True, truncation=True, return_tensors="pt")

# Dataset Class
class BERTDatasetWithFeatures(Dataset):
    def __init__(self, encodings, features, labels):
        self.encodings = encodings
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'features': self.features[idx],
            'labels': self.labels[idx]
        }

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class BERTWithFeatureMLP(nn.Module):
    def __init__(self,  feature_dim, num_labels):
        bert_model_name='aubmindlab/bert-base-arabertv02'
        super(BERTWithFeatureMLP, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)

        # 🧠 MLP branch for features
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        # self.feature_mlp = nn.Sequential(
        #     nn.Linear(feature_dim, 128),
        #     nn.BatchNorm1d(128),
        #     nn.ReLU(),
        #     nn.Linear(128, 64),
        #     nn.BatchNorm1d(64),
        #     nn.ReLU()
        # )


        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size + 64, num_labels)

    def forward(self, input_ids, attention_mask, features):
        # Get BERT CLS output
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [batch, 768]

        # MLP output for features
        feature_output = self.feature_mlp(features)  # [batch, 64]

        # Concatenate
        combined = torch.cat((cls_output, feature_output), dim=1)
        combined = self.dropout(combined)
        return self.classifier(combined)


In [ ]:
train_encodings = tokenizer(train_texts, padding=True, truncation=True, return_tensors="pt")
val_encodings = tokenizer(dev_texts, padding=True, truncation=True, return_tensors="pt")

train_dataset = BERTDatasetWithFeatures(train_encodings, train_features, train_labels)
val_dataset = BERTDatasetWithFeatures(val_encodings, dev_features, dev_labels)

# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)

val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
feature_dim

52

In [ ]:
num_labels = all_df['RL_num_19'].nunique()

In [ ]:


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTWithFeatureMLP(feature_dim=feature_dim, num_labels=num_labels).to(device)

# Freeze BERT
for param in model.bert.parameters():
    param.requires_grad = False

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# Define metric functions
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

def macro_precision(y_true, y_pred):
    return precision_score(y_true, y_pred, average='macro')

def macro_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, average='macro')

def acc(y_true, y_pred):
    return accuracy_score(y_true, y_pred)



In [ ]:
from tqdm import tqdm

EPOCHS = 4
freeze_epochs = 2
for epoch in range(EPOCHS):
    print(f"\n🌟 Epoch {epoch+1}/{EPOCHS}")
    # freezing
    if epoch == freeze_epochs:
      print("🧠 Unfreezing BERT layers...")
      for param in model.bert.parameters():
          param.requires_grad = True
    model.train()
    total_loss = 0

    train_iter = tqdm(train_loader, desc="🔁 Training", leave=False)
    for batch in train_iter:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        train_iter.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)
    print(f"  🔧 Average Training Loss: {avg_train_loss:.4f}")

    # Evaluation
    model.eval()
    val_preds, val_labels_all = [], []

    val_iter = tqdm(val_loader, desc="🔍 Evaluating", leave=False)
    with torch.no_grad():
        for batch in val_iter:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask, features)
            _, preds = torch.max(outputs, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels_all.extend(labels.cpu().numpy())

    # Metrics
    f1 = macro_f1(val_labels_all, val_preds)
    precision = macro_precision(val_labels_all, val_preds)
    recall = macro_recall(val_labels_all, val_preds)
    accuracy = acc(val_labels_all, val_preds)

    print(f"  📊 Val Accuracy:        {accuracy:.4f}")
    print(f"  🎯 Macro F1 Score:      {f1:.4f}")
    print(f"  🧠 Macro Precision:     {precision:.4f}")
    print(f"  💡 Macro Recall:        {recall:.4f}")



🌟 Epoch 1/4


  🔧 Average Training Loss: 2.2226


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


  📊 Val Accuracy:        0.3221
  🎯 Macro F1 Score:      0.0987
  🧠 Macro Precision:     0.1178
  💡 Macro Recall:        0.1147

🌟 Epoch 2/4


  🔧 Average Training Loss: 1.9926


...

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


  📊 Val Accuracy:        0.4993
  🎯 Macro F1 Score:      0.3555
  🧠 Macro Precision:     0.3819
  💡 Macro Recall:        0.3771

🌟 Epoch 4/4


  🔧 Average Training Loss: 1.2276


  📊 Val Accuracy:        0.5325
  🎯 Macro F1 Score:      0.3748
  🧠 Macro Precision:     0.4008
  💡 Macro Recall:        0.3929


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
test_encodings = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt")

test_dataset = BERTDatasetWithFeatures(test_encodings, test_features, test_labels)

test_loader = DataLoader(test_dataset, batch_size=16)


In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="🚀 Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, features=features)
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())


🚀 Predicting: 100%|██████████| 455/455 [00:15<00:00, 30.33it/s]


In [ ]:
all_preds = [label + 1 for label in all_preds]


In [ ]:
test_df["Predicted_RL"] = all_preds
test_df.to_csv("/content/drive/MyDrive/MBZ/Thesis/bert with features/test_predictions_allfeatsNN_freezing_nonorm.csv", index=False)
print("✅ Predictions saved to test_predictions.csv")


<ipython-input-20-934cb019314d>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["Predicted_RL"] = all_preds


✅ Predictions saved to test_predictions.csv


In [ ]:
acc(val_labels_all, val_preds)

0.5324675324675324

In [ ]:
val_preds = [label + 1 for label in val_preds]

In [ ]:
dev_df['Predicted_RL'] = val_preds
dev_df.to_csv("/content/drive/MyDrive/MBZ/Thesis/bert with features/dev_predictions_relu_allfeatsNN_freezing_nonorm..csv", index=False)
print("✅ Predictions saved to dev_predictions.csv")

In [ ]:
# !pip install huggingface_hub
# !huggingface-cli login


In [ ]:
# !huggingface-cli repo create my-hybrid-arabert-model-11-levels


In [ ]:
# from huggingface_hub import HfApi, HfFolder, Repository, create_repo
# import os
# import torch

# # Define repo
# repo_name = "my-hybrid-arabert-model-11-levels"
# local_dir = f"/content/drive/MyDrive/MBZ/Thesis/bert with features/{repo_name}"
# hf_username = "Noorrabie"  # replace with your HF username

# # Create the repo on HF
# HfApi().create_repo(repo_id="Noorrabie/my-hybrid-arabert-model-11-levels", exist_ok=True)


# # Save your model checkpoint
# model_path = os.path.join(local_dir, "pytorch_model.bin")
# os.makedirs(local_dir, exist_ok=True)
# torch.save(model.state_dict(), model_path)

# # Save config (custom example)
# config = {
#     "model_type": "bert-hybrid",
#     "feature_dim": feature_dim,
#     "hidden_size": model.bert.config.hidden_size,
#     "num_labels": num_labels,
#     "base_model": "aubmindlab/bert-base-arabertv02"
# }
# import json
# with open(os.path.join(local_dir, "config.json"), "w") as f:
#     json.dump(config, f)

# # (Optional) Add README
# with open(os.path.join(local_dir, "README.md"), "w") as f:
#     f.write("# Hybrid AraBERT Model\nThis model combines AraBERT with linguistic features for levels 1-11.")

# # Link local dir to HF repo and push
# repo_url = f"{hf_username}/{repo_name}"
# repo = Repository(local_dir=local_dir, clone_from=repo_url)
# repo.push_to_hub()
